# Lab 9: Transfer Learning with BERT

In this lab we are going to be looking at fine-tuning a BERT model to carry out a sequence classification task.

We are going to load in some text from a small number of books in the Gutenberg corpus and see if we can train a classifier to classify which book a piece of text is from 

First lets pull up all of the filenames available.

In [ ]:
import os,random,math
TRAINING_DIR="../../week2/lab2/lab2resources/sentence-completion/Holmes_Training_Data" #this needs to be the parent directory for the training corpus

filenames=os.listdir(TRAINING_DIR)
n=len(filenames)
print("There are {} files in the training directory: {}".format(n,TRAINING_DIR))

#print(filenames)

We are going to create a Book class to store the text from a class and do some very basic pre-processing.  We need to 
* load in the text line-by-line
* get rid of header lines (the stuff in the file before the line which starts \*END\*THE SMALL PRINT!)
* make chunks of text which are longer than 1 line.  These should be easier to classify.  We will try to get sentences - but some chunks may contain multiple sentences.  We are not going to worry about this here.
* return some labelled data split randomly between training and testing

In [ ]:
class Book():
    TRAINING_DIR="../../week2/lab2/lab2resources/sentence-completion/Holmes_Training_Data"
    header_end="*END*THE SMALL PRINT!"
    seed=53
   
    def __init__(self,filename,label=""):
        self.filename=filename
        self.loadfile()
        self.make_chunks()
        if label=="":
            self.label=self.filename
        else:
            self.label=label
        
    def loadfile(self):
        filepath=os.path.join(Book.TRAINING_DIR,self.filename)
        self.lines=[]
        beyond_header=False
        try:
            with open(filepath) as instream:
                for line in instream:
                    line=line.rstrip()
                    
                    if len(line)>0 and beyond_header:
                        self.lines.append(line)
                    if line.startswith(Book.header_end):
                        beyond_header=True
        except UnicodeDecodeError:
            print(f"UnicodeDecodeError processing {filepath}")
    
    def length(self):
        return len(self.chunks)
    
    def head(self,n=10):
        return self.chunks[:n]
    
    def make_chunks(self):
        self.chunks=[]
        
        current=""
        for line in self.lines:
            current+=line+" "
            if line.endswith("."):
                self.chunks.append(current.rstrip())
                current=""
            
    def get_labelled_data(self,split=0.8):
        labelled_data=[(chunk,self.label) for chunk in self.chunks]
        random.seed(Book.seed)
        random.shuffle(labelled_data)
        index=int(self.length()*split)
        return (labelled_data[:index],labelled_data[index:])
                

### Exercise 1
- Create an instance of a Book() and store it in the variable `emma`.  The filename is `EMMA10.TXT` and the label should be `Emma`
- Check the number of sentences = 2028 and have a look at the first 10 sentence
- Repeat to create `ivanhoe` to store the text from `IVNHO12.TXT` with the label `Ivanhoe`.  
- The number of sentences in `ivanhoe` should be 1743

### Exercise 2
- Use the `get_labelled_data()` method to get a training and testing portion from each book (split = 80%).  
- Create 2 pandas dataframes
    - 1 dataframe called `training_df` with all of the training data (from both books)
    - 1 dataframe called  `testing_df`  with all of the test data (from both books).  
    - The columns of both dataframes should be `text` and `label`

## Finetuning BERT to tell the difference between sentences from each book

Now we are going to look at building a classifier on top of BERT.  The first thing we need to do is map the informative label names we have (`Emma` and `Ivanhoe`) to integers which will be used by BERT.  In this simple case, we could just create a dictionary manually.  However, the code below will make a sorted list (without duplicates) of all of the labelnames in the two dataframes and then generate a dictionary which maps each label name to an integer.


In [ ]:
#first we need a map for the labels

#make a list of all of the unique labels in the training and testing dataframes
#sorting it means that it will also be in the same order (alphabetical) rather than depending on order in the training / testing data
labellist=sorted(list(set(training_df['label'].unique()).union(set(testing_df['label'].unique())))) 

labels={label:i for i,label in enumerate(labellist)}
labels

### Exercise 3
Write some code to create a reverse index for the labels which maps the numbers back to the more informative strings

This should result in something which looks as follows:

```
reverse_index={0:'Emma',1:'Ivanhoe'}
```

But obviously, you should create it automatically from the labels dictionary rather than typing it in!

Now we need to store the data in a Dataset class.  This inherits properties from torch's Dataset class and is what is expected.  It is initialised by giving it dataframe from which it can extract a list of labels and a list of texts.   It handles preprocessing including adding CLS and SEP tokens at the beginning and end, tokenization, lower-casing truncation and padding.  It also provides a `\_\_getitem\_\_()` method which allows the Dataset to be indexed into like a list i.e., `myDataset[3]` will return a pair which is the label and text with index 3.

In [ ]:
import torch
import numpy as np
from transformers import BertTokenizer
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')

class Dataset(torch.utils.data.Dataset):
    
    def __init__(self,df,column='text'):
        self.labels=[labels[label] for label in df['label']]
        self.texts=[tokenizer(text.lower(),padding='max_length',max_length=512,truncation=True,return_tensors="pt") for text in df[column]]
        
    def classes(self):
        return self.labels
    
    def __len__(self):
        return len(self.labels)
    
    def get_batch_labels(self,idx):
        return np.array(self.labels[idx])
    
    def get_batch_texts(self,idx):
        return self.texts[idx]
    
    def __getitem__(self,idx):
        batch_texts=self.get_batch_texts(idx)
        batch_y=self.get_batch_labels(idx)
        
        return batch_texts,batch_y
    

train_data=Dataset(training_df)
test_data=Dataset(testing_df)

Lets have a look at one of the dataset items.

The \_\_getitem\_\_ method allows us to index into the dataset and get a particular item

In [ ]:
train_data[0]

We can also just look at the text (or label) for an item as follows.  Note that the tokens have been replaced by their indices in the BERT wordpiece vocabulary.

In [ ]:
train_data.texts[0]

### Exercise 4
Can you turn the token ids back into subwords for one of the inputs?

Now we need to prepare the inputs for the particular device (GPU or CPU) that the model is going to be run on.  Let's just check first whether GPU / CUDA has been enabled.

In [ ]:
use_cuda=torch.cuda.is_available()
if use_cuda:
  print("GPU acceleration enabled")
else:
  print("GPU acceleration NOT enabled.  If using Colab, have you changed the runtype type and selected GPU as the hardware accelerator?")
device=torch.device("cuda" if use_cuda else "cpu")

In [ ]:
def prepare_inputs(input1,label,device):
  label=label.to(device)
  mask=input1['attention_mask'].to(device)
  input_id=input1['input_ids'].squeeze(1).to(device)
  return (input_id,mask,label)

Lets try preparing some inputs and running them through BERT.  We will use the torch DataLoader to manage iterating over the datasets during training and testing.  Here we will just process the first item produced by the DataLoader to see the output from the pre-trained BERT model.

In [ ]:
from transformers import BertModel

train_dataloader=torch.utils.data.DataLoader(train_data,batch_size=2,shuffle=True)
bert=BertModel.from_pretrained('bert-base-uncased')
for train_input,train_label in train_dataloader:
    input_id,mask,label=prepare_inputs(train_input,train_label,device)
    output=bert(input_ids=input_id,attention_mask=mask,return_dict=False)
    break

print(input_id,mask,label)
print(len(output))
output[1]

Now, we need to construct our classification network.  Look at the code below and then answer Exercise 5

In [ ]:
#now we need to put a simple classification layer on top of BERT

from torch import nn
from transformers import BertModel

class BertClassifier(nn.Module):
    
    def __init__(self,dropout=0.5,num_classes=2):
        
        super(BertClassifier,self).__init__()
        
        self.bert=BertModel.from_pretrained('bert-base-uncased')
        self.dropout=nn.Dropout(dropout)
        self.linear=nn.Linear(768,num_classes)
        self.relu=nn.ReLU()
        
    def forward(self,input_id,mask):
        
        last_hidden_layer,pooled_output = self.bert(input_ids=input_id,attention_mask=mask,return_dict=False)
        dropout_output=self.dropout(pooled_output)
        linear_output=self.linear(dropout_output)
        final_layer=self.relu(linear_output)
        
        return final_layer

### Exercise 5

Use the definitions of the initialisation method and the forward method of the BertClassifier class to sketch out what the neural network architecture looks like.

What do you understand by the terms:

- pooled output

- dropout layer

- linear layer

- relu layer


Now we will define a function which will carry out the training of the network.  It will handle
- setting up the DataLoaders
- preparing the inputs for CPU / GPU
- carrying out training and validation for a number of epochs.  In each epoch:
    - iterate over the training data in batches
    - get the output for each input and compute the batch loss (Cross Entropy)
    - use the batch loss to carry out optimisation
    - compute the accuracy and total loss for the training data
    - iterate over the testing / validation data in batches
    - get the output for each input and compute the batch loss
    - compute the accuracy and total loss for the validation data
    - output stats

In [ ]:
#we now need a training loop

from torch.optim import Adam
from tqdm import tqdm



def train(model, train_data,val_data,learning_rate,epochs):
    
    train_dataloader=torch.utils.data.DataLoader(train_data,batch_size=2,shuffle=True)
    val_dataloader=torch.utils.data.DataLoader(test_data,batch_size=2)
    
    use_cuda=torch.cuda.is_available()
    device=torch.device("cuda" if use_cuda else "cpu")
    
    criterion=nn.CrossEntropyLoss()
    optimizer=Adam(model.parameters(),lr=learning_rate)
    
    if use_cuda:
        model=model.cuda()
        criterion=criterion.cuda()
        
    for epoch_num in range(epochs):
        total_acc_train=0
        total_loss_train=0
        model.train()
        for train_input,train_label in tqdm(train_dataloader):
            
            input_id,mask, train_label=prepare_inputs(train_input,train_label,device)
            
            output=model(input_id,mask)
            
            batch_loss=criterion(output,train_label.long())
            total_loss_train +=batch_loss.item()
            
            acc=(output.argmax(dim=1)==train_label).sum().item()
            total_acc_train+=acc
            
            model.zero_grad()
            batch_loss.backward()
            optimizer.step()
            
        total_acc_val=0
        total_loss_val=0
        model.eval()
        with torch.no_grad():
            for val_input,val_label in val_dataloader:
                
                input_id,mask, val_label=prepare_inputs(val_input,val_label,device)
                
                output=model(input_id,mask)
                
                batch_loss=criterion(output,val_label.long())
                
                total_loss_val+=batch_loss.item()
                
                acc=(output.argmax(dim=1)==val_label).sum().item()
                total_acc_val+=acc
                
        print(f'Epochs: {epoch_num+1} | Train Loss: {total_loss_train / len(train_data):.3f} | Train Accuracy: {total_acc_train/len(train_data):.3f}')
        print(f'Val loss: {total_loss_val/len(val_data):.3f} | Val Accuracy: {total_acc_val / len(val_data):.3f}')
        

Here, we define the number of epochs we are going to train for, the learning rate and an instance of our BertClassifier network.

In [ ]:
EPOCHS=1  
model=BertClassifier(num_classes=len(labels.keys()))
LR=1e-6


In [ ]:
model=BertClassifier(num_classes=len(labels.keys()))

#this will freeze the pre-trained BERT model and just make the classification head trainable
#can speed things up and avoid "catastrophic forgetting" / overfitting on task-specific data
model.bert.requires_grad_(False) 

Now we are actually going to train the model.  This might take some time - particularly if you are running on a CPU (1.5 hrs per epoch on my laptop!).  I recommend running on colab with GPU enabled (or using the lab machines with GPU).

In [ ]:
train(model,train_data,test_data,LR,EPOCHS)

We need to be able to save the model to be able to use it elsewhere (without training again!)

In [ ]:
#I am not going to save my model as I didn't wait for it to finish training!

#output_dir="bert-base-uncased-bookclassifier"
#torch.save(model,output_dir)

We can load it up like this.  This could be in another notebook.  If loading in another notebook, you should make sure the BertClassifier class is also defined in that notebook (along with other necessary imports).

In [ ]:
#I am going to load a model I trained on colab!

input_dir="Lab9resources/julie-bert-base-uncased-bookclassifier"
complete_model=torch.load(input_dir)

Here's an evaluation loop we can use to evaluate on some test data.  We also return the predictions which can than be added to the dataframe

In [ ]:
batchsize=2
def evaluate(model,test_dataset):
    model.eval()
    test_dataloader=torch.utils.data.DataLoader(test_dataset,batch_size=batchsize)
    
    use_cuda=torch.cuda.is_available()
    device=torch.device("cuda" if use_cuda else "cpu")
    
    if use_cuda:
        model=model.cuda()
        
    total_acc_test=0
    with torch.no_grad():
        count=0
        predictions=[]
        for test_input,test_label in tqdm(test_dataloader):
            count+=batchsize
            test_label=test_label.to(device)
            mask=test_input['attention_mask'].to(device)
            input_id=test_input['input_ids'].squeeze(1).to(device)
            output=model(input_id,mask)
            #print(output.argmax(dim=1),test_label)
            predictions.append(output.argmax(dim=1))  #save the prediction for further analysis
            acc=(output.argmax(dim=1)==test_label).sum().item()
            
            total_acc_test+=acc
            if count%100==0:
                print(f'Accuracy so far = {total_acc_test/count: .3f}')
            
    print(f'Test accuracy: {total_acc_test/len(test_dataset): .3f}')
    return predictions

This takes around 6 minutes to run on my laptop.  I have the evaluation loop printing out "Accuracy so far" as I get bored waiting to the end to see the results - it also gives a very rough indication of how stable the method is

In [ ]:
predictions=evaluate(model, test_data)

### Exercise 6
Add the predicted label for each test item to the dataframe with the test data.

### Exercise 7
Compute the confusion matrix / precision and recall scores for the different classes.  What does this analysis tell you about the errors?

Precision is better for Ivanhoe whereas recall is better for Emma.  This suggests that the classifier is over-predicting Emma - this is consistent with there being more training data for Emma.

## Extension
Now carry out Lab 9b on sequence labelling with BERT.
